In [1]:
import sys
sys.path.append("src")

In [2]:
import os
import json
import string

import chromadb

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from rag.vector_db import ChromaDB
from rank_bm25 import BM25Okapi

# Test DB

In [3]:
client = chromadb.HttpClient(host="localhost", port=8000)
collection = client.get_or_create_collection(name="rag-collection")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/auth/identity "HTTP/1.1 200 OK"
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"


# Data

In [4]:
sample_txt_data = "data/news_articles/05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt"

with open(sample_txt_data, "r") as f:
    file = f.read()

In [5]:
mami_bubu = "data/mami_bubu/"

data = []

for filename in os.listdir(mami_bubu):
    file_path = os.path.join(mami_bubu, filename)
    with open(file_path, "r") as f:
        file = json.load(f)

    data.append({
        "id": filename,
        "text": str(file)
    })

In [6]:
type(data[0]["text"])

str

In [7]:
for d in data:
    print(len(d["text"]))

3212
542
1854
3292
3013
552
7167
7317
1775
1395
818
6422
4502
2154
4085
542
2989
2933
1862
1394
542
3546
1796
2398
1814
3003
1809
3275
2257
3213
2399
2297
792
3825
552
1796
1788
1829
2087
1796
5801
2257
552
1820
436
1775
1796
5359
805
2440
682
1844
590


# vectorizer

In [8]:
# model = SentenceTransformer(
#     "mixedbread-ai/mxbai-embed-large-v1",
#     truncate_dim=512
# )
# model.save_pretrained("weights/mxbai-embeddings/")

In [81]:
model = SentenceTransformer("weights/mxbai-embeddings/", truncate_dim=512)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: weights/mxbai-embeddings/
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'passage']


In [10]:
model.encode(data[0]["text"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([-0.284806  , -0.07760914,  0.19776972, ..., -0.05770492,
       -0.69765437, -0.26364008], shape=(1024,), dtype=float32)

In [ ]:
cccc

# hybrid search

In [11]:
import re

In [12]:
db = ChromaDB(collection_name="rag-collection")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/auth/identity "HTTP/1.1 200 OK"
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"


In [13]:
sample_query = "ara casual button set"

query = model.encode(sample_query)
query

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([-0.28106284,  0.8760762 , -0.21192123, ...,  0.21943468,
       -0.05419864, -0.98622423], shape=(1024,), dtype=float32)

## BM25

In [69]:
def _tokenized(text: str):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    text = text.split(" ")
    return text

def _cleaning(text: str):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    
    return text

In [70]:
corpus = [_cleaning(d["text"]) for d in data]

tokenized_corpus = list(map(_tokenized, corpus))

print(corpus[0])
print(tokenized_corpus[0])

nama produk ara casual button set setelan anak bayi 1 tahun size  deskripsi set atasan dan bawahan yang terdiri dari kaos lengan pendek dan celana panjang terbuat dari bahan katun yang lembut dan nyaman untuk dipakai si kecil seharihari harga retail 490000 harga reseller diskon 15 416500 stockist diskon 30 343000 berat gram 1000 kategori setelan baju anak perempuan jumlah varian 15 varian warna almond gambar httpsngorder1sgp1digitaloceanspacescom102804productscasualbuttonsetsetelananakbayi1tahun1678073508073jpg jumlah stok real 360 warna army gambar httpsngorder1sgp1digitaloceanspacescom102804productscasualbuttonsetsetelananakbayi1tahun1668134390222jpg jumlah stok real 293 warna baby tosca gambar httpsngorder1sgp1digitaloceanspacescom102804productscasualbuttonsetsetelananakbayi1tahun1675817895056jpg jumlah stok real 502 warna black gambar httpsngorder1sgp1digitaloceanspacescom102804productsaracasualbuttonsetsetelananakbayi1tahun1700536697474png jumlah stok real 367 warna brick gambar h

In [71]:
bm25 = BM25Okapi(tokenized_corpus)

In [72]:
scores = bm25.get_scores(_tokenized(sample_query))
print(type(scores))
scores

<class 'numpy.ndarray'>


array([3.89965958, 0.        , 0.        , 3.88278291, 0.        ,
       0.        , 0.        , 0.26125212, 3.28099519, 0.        ,
       5.33799041, 0.18675875, 0.23607307, 1.02173304, 0.21889797,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.35977661, 2.11778449, 0.        , 0.        ,
       0.        , 0.        , 3.88278291, 1.02173304, 3.89965958,
       0.        , 0.        , 4.77033665, 0.26665121, 0.        ,
       2.11778449, 3.28099519, 0.        , 0.        , 3.28099519,
       0.20325721, 1.02173304, 0.        , 0.        , 0.        ,
       3.28099519, 2.11778449, 0.19760935, 4.75527612, 0.        ,
       4.77033665, 0.        , 0.        ])

In [73]:
# get top 10
idx = scores.argsort()[-10:][::-1]
idx

array([10, 32, 50, 48, 29,  0, 27,  3,  8, 36])

In [74]:
candidates = [data[i] for i in idx]
candidates

[{'id': 'Ara Casual Button Motif 6-12 Bulan.json',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 6-12 Bulan', 'Size': '-', 'Deskripsi': 'casual button motif', 'Harga Retail': 74900.0, 'Harga Reseller (Diskon 15%)': 63665.0, 'Stockist (Diskon 30%)': 52430.0, 'Berat (gram)': 125.0, 'Kategori': 'Setelan Baju Anak Unisex', 'jumlah varian': 3, 'varian': [{'Warna': 'Mangga', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021789349.png', 'Jumlah Stok Real': 43}, {'Warna': 'Manggis', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021784556.png', 'Jumlah Stok Real': 37}, {'Warna': 'Rambutan', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021781664.png', 'Jumlah Stok Real': 39}]}"},
 {'id': 'Ara Casual Button Motif 1 Tahun.json',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 1 Tahun', 'Size'

In [78]:
[scores[i] for i in idx]

[np.float64(5.337990406147155),
 np.float64(4.7703366519446595),
 np.float64(4.7703366519446595),
 np.float64(4.755276120045967),
 np.float64(3.8996595815084554),
 np.float64(3.8996595815084554),
 np.float64(3.882782911590387),
 np.float64(3.882782911590387),
 np.float64(3.2809951875956616),
 np.float64(3.2809951875956616)]

In [77]:
ranked_result = []
for i in idx:
    result = {
        "id": data[i]["id"],
        "text": data[i]["text"],
        "score": scores[i]
    }
    ranked_result.append(result)

ranked_result

[{'id': 'Ara Casual Button Motif 6-12 Bulan.json',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 6-12 Bulan', 'Size': '-', 'Deskripsi': 'casual button motif', 'Harga Retail': 74900.0, 'Harga Reseller (Diskon 15%)': 63665.0, 'Stockist (Diskon 30%)': 52430.0, 'Berat (gram)': 125.0, 'Kategori': 'Setelan Baju Anak Unisex', 'jumlah varian': 3, 'varian': [{'Warna': 'Mangga', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021789349.png', 'Jumlah Stok Real': 43}, {'Warna': 'Manggis', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021784556.png', 'Jumlah Stok Real': 37}, {'Warna': 'Rambutan', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021781664.png', 'Jumlah Stok Real': 39}]}",
  'score': np.float64(5.337990406147155)},
 {'id': 'Ara Casual Button Motif 1 Tahun.json',
  'text': "{'Nama Produk':

In [49]:
ccccccc

NameError: name 'ccccccc' is not defined

# dense search

In [55]:
for i, d in enumerate(corpus):
    embedding = model.encode(d)
    data[i]["embedding"] = embedding

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
for d in data:
    db.insert_vectors(
        ids = [d["id"]],
        docs = [d["text"]],
        embeddings = [d["embedding"]]
    )

INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_

In [57]:
db.search_vectors(
    query = query,
    top_k = 5
)

INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f92c4c65-ce68-4af3-ab97-14b4f0066688/query "HTTP/1.1 200 OK"


{'ids': [['Ara Casual Button Set (Setelan anak Bayi 1 Tahun).json',
   'Ara Casual Button Set (Setelan anak Bayi 2 Tahun).json',
   'Ara Casual Button Set (Setelan anak Bayi 0 - 6 Bulan).json',
   'Ara Casual Button Set (Setelan anak Bayi 6 - 12 Bulan).json',
   'Ara Casual Button Motif 0-6 Bulan.json']],
 'distances': [[76.68739318847656,
   76.72421264648438,
   79.15571594238281,
   79.82957458496094,
   80.98141479492188]],
 'embeddings': None,
 'metadatas': [[None, None, None, None, None]],
 'documents': [["{'Nama Produk': 'Ara Casual Button Set (Setelan anak Bayi 1 Tahun)', 'Size': '-', 'Deskripsi': 'Set atasan dan bawahan yang terdiri dari kaos lengan pendek dan celana panjang. Terbuat dari bahan katun yang lembut dan nyaman untuk dipakai si kecil sehari-hari.', 'Harga Retail': 49000.0, 'Harga Reseller (Diskon 15%)': 41650.0, 'Stockist (Diskon 30%)': 34300.0, 'Berat (gram)': 100.0, 'Kategori': 'Setelan Baju Anak Perempuan', 'jumlah varian': 15, 'varian': [{'Warna': 'Almond', 'Ga